# UWA AI Club — AI Agents Workshop

## Build from prompt → context → tools → agents → multi-agent → RAG

This notebook is deliberately **not** an API tutorial. The infrastructure is already set up so you can focus on the decisions that make an agent useful.

### What you will edit
Look for **✏️ YOUR TURN** cells. Most other cells should just be run.

### What the trace shows
The workshop trace shows **observable actions** — user messages, tool requests, tool results, errors and final answers. It does not ask for or expose hidden chain-of-thought.


## 00 — Setup 🔒

Run the next two cells once. If your environment already has the packages installed, the install cell will finish quickly.


In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from getpass import getpass
import os

if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Workshop Anthropic API key: ")

# Change this one line if your workshop key has access to a different model.
os.environ.setdefault("WORKSHOP_MODEL", "claude-sonnet-5")

from workshopkit import *
print("Ready. Model:", DEFAULT_MODEL)


---
# Mission 1 — Prompt Surgery

Your starting prompt is intentionally weak:

> Tell me about cybersecurity.

Rewrite it so that a specific user could judge whether the answer is useful. Think about **success criteria, task, context, examples when useful, constraints, and output**.


In [ ]:
# ✏️ YOUR TURN
PROMPT = """
Tell me about cybersecurity.
"""

print(ask_claude(
    "You are a helpful assistant. Follow the user's request precisely.",
    PROMPT,
    max_tokens=900,
))


### Quick checkpoint
Before moving on, ask your neighbour:

- What user are you helping?
- What does success look like?
- What information did you add that actually changes the answer?
- What would you test in the output?


---
# Mission 2 — Your First Live Model Call

Turn Claude into a useful **student event planning assistant**. You control the system instructions and the user request.


In [ ]:
# ✏️ YOUR TURN
SYSTEM_PROMPT = """
You are ...
"""

USER_REQUEST = """
Help me plan a beginner AI workshop for university students.
"""

print(ask_claude(SYSTEM_PROMPT, USER_REQUEST, max_tokens=1000))


---
# Mission 3 — Give the Agent One Tool

Claude does **not** know the workshop's simulated weather data. It must decide whether to call the `get_weather` tool.

The function, schema and tool loop are already implemented.


In [ ]:
# 🔒 Inspect the tool contract
print(get_weather.api_definition())


In [ ]:
# ✏️ YOUR TURN
AGENT_INSTRUCTIONS = """
You help UWA AI Club make practical event decisions.
Use available tools when the task depends on information you do not already have.
Give a concise recommendation and explain which live facts matter.
"""

TASK = "Should we run the club BBQ outside this Saturday in Perth?"

result = run_agent(
    AGENT_INSTRUCTIONS,
    TASK,
    tools=[get_weather],
    max_steps=4,
)
show_trace(result)


### What to notice
The important sequence is:

**request → tool selection → structured arguments → tool result → updated answer**

Claude did not execute the Python function itself. It requested the tool; the workshop runtime executed it and returned the result.


---
# Mission 4 — Give Marv a Toolbox

Now the agent can use five capabilities. Your job is **not** to tell it a rigid sequence. Your job is to write instructions that make it choose useful actions without calling everything unnecessarily.


In [ ]:
# 🔒 Available tools
for tool in BASIC_TOOLS:
    print(f"{tool.name}: {tool.description}")


In [ ]:
# ✏️ YOUR TURN
AGENT_INSTRUCTIONS = """
You are ...

Your goal is ...

Use tools when ...

Before finishing, make sure ...
"""

TASK = """
Organise a 60-person beginner AI Agents workshop next Friday.
The total budget must stay under $700.
Recommend a viable room and plan, then create only the most important preparation tasks.
"""

result = run_agent(
    AGENT_INSTRUCTIONS,
    TASK,
    tools=BASIC_TOOLS,
    max_steps=8,
)
show_trace(result)


### Compare traces
Find someone near you and compare:

- Did one agent call more tools?
- Did either agent create tasks too early?
- Did it calculate the budget before making a recommendation?
- Which sentence in your instructions seems to have changed the behaviour?


---
# Mission 4B — Let the Environment Push Back

Agents become interesting when the path changes because the world changes.

Run the next cell to deliberately make `find_room` fail, then run your previous agent again. Inspect how it handles the error.


In [ ]:
# 🔒 Toggle a simulated external-service failure
os.environ["WORKSHOP_SIMULATE_ROOM_FAILURE"] = "1"
print("Room booking failure simulation: ON")


In [ ]:
# ✏️ YOUR TURN — reuse or improve your instructions
FAILURE_TASK = """
We need a room for 60 students next Friday. Work out a safe plan.
If an external tool fails, do not pretend it succeeded.
"""

failure_result = run_agent(
    AGENT_INSTRUCTIONS,
    FAILURE_TASK,
    tools=BASIC_TOOLS,
    max_steps=6,
)
show_trace(failure_result)


In [ ]:
# 🔒 Turn the failure off for later missions
os.environ.pop("WORKSHOP_SIMULATE_ROOM_FAILURE", None)


---
# Mission 5 — Build an Orchestrator

We now have three specialist agents:

- `research_agent`
- `budget_agent`
- `audience_agent`

Each specialist is a **real separate Claude call** with its own system instructions. The manager sees them as tools and decides what to delegate.

Your job: write the manager's instructions.


In [ ]:
# 🔒 Specialist contracts
for specialist in SPECIALISTS:
    print(f"{specialist.name}: {specialist.description}")


In [ ]:
# ✏️ YOUR TURN
MANAGER_PROMPT = """
You are the lead decision-maker for UWA AI Club.

Delegate work when ...
Do not delegate when ...
When specialists return results ...
Your final recommendation must ...
"""

MULTI_AGENT_TASK = """
Choose ONE event for next month:
1. AI Hackathon
2. AI Careers Night
3. AI Agents Workshop

Consider evidence, likely student value, complexity and budget.
State assumptions and produce one recommendation.
"""

manager_result = run_manager(MANAGER_PROMPT, MULTI_AGENT_TASK, max_steps=7)
show_trace(manager_result, "MULTI-AGENT TRACE")


### Architecture checkpoint
Was multi-agent actually useful here?

Think about the coordination cost: each specialist is another model call. If one normal model call would solve the task reliably, a multi-agent system may be unnecessary.


---
# Mission 6 — RAG / Private Knowledge

Claude should not guess private club policy. The `search_club_docs` tool searches the local workshop documents and returns relevant passages with source filenames.

The retrieval algorithm is intentionally simple so we can focus on the **architecture**. In a production system, retrieval might use embeddings, hybrid search, reranking, permissions and metadata filters.


In [ ]:
# 🔒 Try retrieval directly first
print(search_club_docs.execute({"query": "in-kind sponsorship gift card event consumables", "top_k": 3}))


In [ ]:
# ✏️ YOUR TURN
RAG_AGENT_INSTRUCTIONS = """
You answer questions about UWA AI Club workshop policy.
When a question depends on private club rules, retrieve relevant documents instead of guessing.
Ground your answer in the retrieved passages and name the source file(s) you relied on.
If the documents do not answer the question, say so.
"""

RAG_TASK = """
A sponsor cannot give us unrestricted cash. Can we ask them to support event snacks using a prepaid purchasing method instead, and what should we record?
"""

rag_result = run_agent(
    RAG_AGENT_INSTRUCTIONS,
    RAG_TASK,
    tools=[search_club_docs],
    max_steps=5,
)
show_trace(rag_result, "RAG AGENT TRACE")


---
# Mission 7 — Build Your Own Agent

Choose a real problem you care about. Keep the architecture as simple as you can.

A good starting template:

1. **Goal** — what outcome should the system achieve?
2. **Context** — what does it need to know now?
3. **Tools** — what must it be able to do or retrieve?
4. **Control** — workflow or agent?
5. **Knowledge** — does it need private/current information?
6. **Approval** — which actions should stop for a person?


In [ ]:
# ✏️ YOUR TURN
MY_AGENT_INSTRUCTIONS = """
You are ...
Your goal is ...
Use tools when ...
Ask for human approval before ...
Stop when ...
"""

MY_TASK = """
...
"""

MY_TOOLS = [get_weather, find_room, calculate_budget]  # edit this list

my_result = run_agent(MY_AGENT_INSTRUCTIONS, MY_TASK, tools=MY_TOOLS, max_steps=8)
show_trace(my_result, "MY AGENT")


---
# Where to go next

You now have the core mental model:

**prompt → context → model → tools → loop → knowledge → orchestration**

Next things worth exploring after the workshop:

- stronger evaluation of prompts and agent outcomes
- semantic / hybrid retrieval and reranking
- real APIs instead of the workshop's mock tools
- MCP servers and clients
- permission models and durable state
- Claude Managed Agents for managed long-running/asynchronous agent work

The instructor folder contains an MCP v2 demo and a Managed Agents demo kept separate from the core participant notebook because those surfaces evolve faster.
